# POLITE — 2026-07-30 observation

Rapid dual-beam commissioning after tracking proved unavailable: short same-pair bursts, beam/PSF diagnostics, and only a validity-gated double-ratio cycle.

Configuration: V filter; Mode 5, gain 168, offset 20; 1×1. The server remains full-frame by default. This notebook uses a full-frame stellar probe, then a validated session ROI; the mount is never connected, enabled, homed, slewed, or tracked.

## Operating rules

- Use any manually acquired star field; give each saved burst a distinct field label. Never treat unrelated stars as repeated photometry.
- HWP, focuser, and field-rotator moves use the approved palette helpers and their read-backs are logged. The PWI4 mount is never connected, enabled, homed, slewed, tracked, or parked.
- Stop if either beam clips, either beam approaches the edge margin, the timestamp is implausible, the EFW identity gate fails, or the temperature is unstable.
- This notebook captures one supervised frame at a time; do **not** run `execute_night.py` simultaneously or replace a bounded burst with `while True`. A saturated HWP frame invalidates its whole four-angle cycle.

## 0 · Shared preamble

This is the template preamble adapted for the extra `observation_notebooks/` directory. Run these two cells first.

In [ ]:
# --- POLITE path bootstrap ---
import os, sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root:
    _root = _root.parent
if not (_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print('POLITE root:', _root)


In [ ]:
# --- Shared import set ---
%matplotlib inline
from pathlib import Path
import csv
import re

import numpy as np
import matplotlib.pyplot as plt

from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
from obs_utils.roi import CameraROI, apply_roi, camera_effective_area

plt.rcParams['figure.dpi'] = 110


## 1 · Tonight's card

Gain 168 is the as-used setting for this salvage sequence. It does not replace lost photon signal, and its conversion gain/read noise are uncharacterized tonight; use it only with short, unsaturated exposures. Retain an exposure only if both beam peaks are comfortably below the observed digital ceiling and any measured linearity limit.

In [ ]:
NIGHT = '20260730'
SESSION_DIR = Path('FITSDATA') / NIGHT
FILTER_NAME = 'Photometric V'
READOUT_MODE, GAIN, OFFSET = 5, 168, 20  # as-used salvage setting; do not silently revert to the project default
SETPOINT_C = -15.0
BINNING = 1
HWP_ANGLES = (0.0, 45.0, 22.5, 67.5)  # q beam-swap pair, then u pair

BURST_N = 5
BURST_TARGET_1 = 'field_1'  # replace with an operator field label, not a claimed standard
BURST_TARGET_2 = 'field_2'
FOCUS_EXP_S = 0.01       # shorten if moon/background or either beam peak is high
SCIENCE_EXP_S = None     # set after the focused stellar probe
TWILIGHT_FLATS_TAKEN = False  # sky was dark before a valid flat sequence could start
FOCUS_POSITIONS = None      # short nominal-centred sweep, entered after first full-frame probe
FOCUS_SELECTED = None       # human-selected position from the focus curve

SESSION_DIR.mkdir(parents=True, exist_ok=True)
print('session:', SESSION_DIR)


## 2 · Bring up camera, EFW, HWP, focuser, and field rotator

**COMMANDS HARDWARE.** The PWI4 client is used only for the focuser and field rotator; no mount operation is issued. On the observatory PC, start the Alpaca services if needed; then connect components independently.

In [ ]:
# Observatory Windows PC only; skip on the lab Mac.
from obs_utils.alpaca_servers import start_observatory_alpaca_servers
start_observatory_alpaca_servers(
    ascom_endpoint=uc.ALPACA_CONFIG.host,
    qhy_endpoint=uc.ALPACA_CONFIG.camera_host,
)


In [ ]:
# COMMANDS CONNECTIONS ONLY — PWI4 is used only for its auxiliary axes.
s = obs.connect_camera()
s = obs.connect_filter_wheel()
s = obs.connect_hwp()           # HWP path, not the PWI4 field rotator
s = obs.connect_focuser()
s = obs.connect_field_rotator()
s.status()


## 3 · Fail-closed configuration gates

**COMMANDS HARDWARE.** Verify the installed EFW order, set and read back the actual gain-168 configuration, and wait for temperature stability. The opaque Dark slot is mandatory for bias and dark frames because the camera has no shutter.

In [ ]:
from obs_utils.night_safety import verify_filter_wheel, cooler_gate, HWP_DEFAULT_TOL_DEG

verify_filter_wheel(s.imaging)
assert s.camera is not None
s.camera.BinX = BINNING
s.camera.BinY = BINNING
s.camera.Gain = GAIN
s.camera.Offset = OFFSET
s.camera.ReadoutMode = READOUT_MODE
readback = {'BinX': s.camera.BinX, 'BinY': s.camera.BinY, 'Gain': s.camera.Gain,
            'Offset': s.camera.Offset, 'ReadoutMode': s.camera.ReadoutMode}
print('read-back:', readback)
assert readback == {'BinX': BINNING, 'BinY': BINNING, 'Gain': GAIN, 'Offset': OFFSET, 'ReadoutMode': READOUT_MODE}
EFFECTIVE_AREA = camera_effective_area(s.camera)
assert (EFFECTIVE_AREA.startx, EFFECTIVE_AREA.starty, EFFECTIVE_AREA.numx, EFFECTIVE_AREA.numy) == (24, 0, 6252, 4176), EFFECTIVE_AREA
FULL_FRAME_ROI = CameraROI(startx=24, starty=0, numx=6252, numy=4176, binx=1, biny=1)
PROVISIONAL_ROI = CameraROI(startx=524, starty=0, numx=5252, numy=4176, binx=1, biny=1)
FULL_FRAME_ROI.validate(s.camera.CameraXSize, s.camera.CameraYSize, effective_area=EFFECTIVE_AREA)
PROVISIONAL_ROI.validate(s.camera.CameraXSize, s.camera.CameraYSize, effective_area=EFFECTIVE_AREA)
ACTIVE_ROI = apply_roi(s.camera, FULL_FRAME_ROI, effective_area=EFFECTIVE_AREA)
print('QHY effective area:', EFFECTIVE_AREA)
print('active ROI:', ACTIVE_ROI)
s.camera.SetCCDTemperature = SETPOINT_C
achieved_temp = cooler_gate(s.camera, SETPOINT_C, tol_c=0.5, stable_s=30.0, timeout_s=900.0, poll_s=5.0, assume_yes=False, verbose=True)
print(f'cooler accepted at {achieved_temp:+.2f} C')


In [ ]:
# COMMANDS HWP MOTION: prove the stage responds before science.
for requested in HWP_ANGLES:
    achieved = s.hwp(requested)
    assert abs(achieved - requested) <= HWP_DEFAULT_TOL_DEG, (requested, achieved)
    print(f'HWP {requested:5.1f} -> {achieved:7.3f} deg')


## 4 · Capture helpers and plain-CSV ledger

The raw FITS are authoritative. `capture_manifest.csv` is a small operator ledger, not a database. `aperture_measurements.csv` is deliberately long-form so the reduction can retain every beam and aperture radius rather than only a final average.

In [ ]:
from alpyca_tools.fits_writer import DetectorCards, FitsHeaderConfig, PolarimetryCards

def _safe(text):
    return re.sub(r'[^A-Za-z0-9]+', '', str(text))

def _next_sequence():
    seqs = []
    for path in SESSION_DIR.glob('*.fits'):
        match = re.match(r'(\d{8})_', path.name)
        if match:
            seqs.append(int(match.group(1)))
    return max(seqs, default=0) + 1

SEQUENCE = _next_sequence()
MANIFEST = SESSION_DIR / 'capture_manifest.csv'
APERTURE_LEDGER = SESSION_DIR / 'aperture_measurements.csv'

if not MANIFEST.exists():
    with MANIFEST.open('w', newline='') as f:
        csv.writer(f).writerow(['filename', 'block', 'target', 'imagetyp', 'exptime_s',
                               'hwp_requested_deg', 'hwp_achieved_deg', 'instrot_deg',
                               'cycle', 'focus_label', 'startx', 'starty', 'numx', 'numy'])
if not APERTURE_LEDGER.exists():
    with APERTURE_LEDGER.open('w', newline='') as f:
        csv.writer(f).writerow(['filename', 'beam', 'aperture_radius_px', 'annulus_rin_px',
                               'annulus_rout_px', 'aperture_sum_adu', 'sky_adu_per_px',
                               'net_source_adu', 'fwhm_px', 'ellipticity', 'peak_adu', 'flag'])

def set_active_roi(roi, *, require_effective_area=True):
    global ACTIVE_ROI
    ACTIVE_ROI = apply_roi(s.camera, roi, effective_area=EFFECTIVE_AREA if require_effective_area else None)
    print('active ROI:', ACTIVE_ROI)
    return ACTIVE_ROI

def capture_one(*, frame_type, block, exposure_s, target=None, hwp_requested=None,
                hwp_achieved=None, instrot_deg=None, cycle=None, focus_label=None):
    """Capture one provenance-rich frame; never commands the mount."""
    global SEQUENCE
    is_dark = frame_type in {'BIAS', 'DARK'}
    s.filter('Dark' if is_dark else FILTER_NAME)
    name_target = _safe(target) + '_' if frame_type == 'LIGHT' and target else ''
    filename_label = {'BIAS': 'Bias', 'DARK': 'Dark', 'FLAT': 'FlatField', 'LIGHT': 'Light'}[frame_type]
    filename = f'{SEQUENCE:08d}_{name_target}{filename_label}_{_safe(FILTER_NAME)}_{exposure_s:.3f}secs'
    if frame_type == 'FLAT':
        filename += '_1x1'
    filename += '.fits'
    pol = PolarimetryCards(hwp_angle_deg=hwp_achieved, instrument_rotator_deg=instrot_deg,
                           pol_seq_id=block if frame_type == 'LIGHT' else None,
                           pol_seq_index=cycle, hwp_uncert_deg=0.012)
    header = FitsHeaderConfig(
        imagetyp='FLAT' if frame_type == 'FLAT' else frame_type,
        object_name=target if frame_type == 'LIGHT' else None, instrument='QHY268M',
        filter_name='Dark' if is_dark else FILTER_NAME, binx=1, biny=1,
        detector=DetectorCards(gain_setting=GAIN, offset_setting=OFFSET, readout_mode=READOUT_MODE,
                                readout_mode_name='Mode 5', cooler_setpoint_c=SETPOINT_C),
        polarimetry=pol,
        extra_cards={'BLOCK': block, 'HWPREQ': hwp_requested, 'FOCUSPOS': str(focus_label or '')},
    )
    path = Path(s.expose(exposure_s, out_path=SESSION_DIR / filename, dark=is_dark, header=header,
                          gain=GAIN, offset=OFFSET, readout_mode=READOUT_MODE,
                          binx=ACTIVE_ROI.binx, biny=ACTIVE_ROI.biny, startx=ACTIVE_ROI.startx,
                          starty=ACTIVE_ROI.starty, numx=ACTIVE_ROI.numx, numy=ACTIVE_ROI.numy))
    with MANIFEST.open('a', newline='') as f:
        csv.writer(f).writerow([path.name, block, target or '', frame_type, exposure_s, hwp_requested,
                                hwp_achieved, instrot_deg, cycle, focus_label or '', ACTIVE_ROI.startx,
                                ACTIVE_ROI.starty, ACTIVE_ROI.numx, ACTIVE_ROI.numy])
    SEQUENCE += 1
    print(path.name)
    return path


## 5 · Twilight-flat decision

**SKIP TONIGHT.** Sky was already dark at 20:55 before a valid twilight sequence could start. Do not take sky flats, a PTC pair, or polarimetric flats now; no valid dark-sky substitute exists. Missing flats never delay the short star-field diagnostics.

In [ ]:
assert not TWILIGHT_FLATS_TAKEN
print('Twilight flats/PTC: SKIPPED — proceed directly to star-field diagnostics.')


## 6 · Any-star acquisition, ROI refinement, focus, and exposure selection

Acquire any field containing a clean Savart pair manually. The first stellar frame is deliberately full-frame: measure both beam bounding boxes and one inter-frame drift estimate before entering `SCIENCE_ROI`. It must contain both beams plus the planned five-frame burst margin. Run a short nominal-centred PWI4 focuser sweep; inspect both beams in the saved probe and choose the common focus manually. Do not intentionally defocus.

In [ ]:
# MANUAL ACTION: acquire and center both beams of BURST_TARGET_1 at nominal focus, then run this cell.
assert FOCUS_EXP_S is not None and FOCUS_EXP_S > 0
set_active_roi(FULL_FRAME_ROI)
achieved = s.hwp(0.0)
acquisition_path = capture_one(frame_type='LIGHT', block='field_fullframe_acquisition', exposure_s=FOCUS_EXP_S, target=BURST_TARGET_1,
                               hwp_requested=0.0, hwp_achieved=achieved, focus_label='nominal')
live.frame_report(acquisition_path)


### ROI checkpoint

Inspect the full-frame acquisition. After confirming both beam boxes and the drift margin fit inside the 500-column crop, run the next cell.

In [ ]:
SCIENCE_ROI = PROVISIONAL_ROI
set_active_roi(SCIENCE_ROI)


### Focus positions

Enter a short nominal-centred set of safe PWI4 positions in the next cell, then run the sweep cell.

In [ ]:
FOCUS_POSITIONS = ()  # replace with at least three safe positions, e.g. (12340, 12360, 12380)
assert len(FOCUS_POSITIONS) >= 3


In [ ]:
achieved = s.hwp(0.0)
sweep = s.focus_sweep(FOCUS_POSITIONS, FOCUS_EXP_S, gain=GAIN, offset=OFFSET, readout_mode=READOUT_MODE,
                      binx=ACTIVE_ROI.binx, biny=ACTIVE_ROI.biny, startx=ACTIVE_ROI.startx, starty=ACTIVE_ROI.starty,
                      numx=ACTIVE_ROI.numx, numy=ACTIVE_ROI.numy)
live.focus_curve(sweep)


### Select focus and science exposure

Inspect both beams and edit `FOCUS_SELECTED` below. After its focused probe, edit `SCIENCE_EXP_S` in the following cell to keep both beam peaks below half the observed ceiling.

In [ ]:
FOCUS_SELECTED = None  # replace with the manually chosen common focus position
assert FOCUS_SELECTED is not None
s.focus(FOCUS_SELECTED)
focused_path = capture_one(frame_type='LIGHT', block='focused_probe', exposure_s=FOCUS_EXP_S, target=BURST_TARGET_1,
                           hwp_requested=0.0, hwp_achieved=achieved, focus_label=str(FOCUS_SELECTED))
live.frame_report(focused_path)


In [ ]:
SCIENCE_EXP_S = None  # replace after inspecting focused_path
assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0


## 7 · Rapid same-pair bursts and optional polarimetry

Take short, bounded five-frame bursts at HWP 0° whenever one Savart pair remains on detector. Its changing centroid is acceptable only when each frame is photometered at its detected position; record its displacement as a systematic. Unrelated fields are assembly diagnostics, not an empirical S/N sequence. Attempt one four-angle cycle only when the same pair is likely to remain inside `SCIENCE_ROI` throughout; never recenter inside that cycle.

In [ ]:
def capture_same_pair_burst(target, block):
    assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0
    achieved = s.hwp(0.0)
    return [capture_one(frame_type='LIGHT', block=block, exposure_s=SCIENCE_EXP_S, target=target,
                        hwp_requested=0.0, hwp_achieved=achieved, cycle=i, focus_label='selected')
            for i in range(BURST_N)]


In [ ]:
def capture_beam_swap_cycle(target, block, cycle, instrot_deg):
    for angle in HWP_ANGLES:
        achieved = s.hwp(angle)
        capture_one(frame_type='LIGHT', block=block, exposure_s=SCIENCE_EXP_S, target=target,
                    hwp_requested=angle, hwp_achieved=achieved, instrot_deg=instrot_deg,
                    cycle=cycle, focus_label='selected')



### Same-pair burst 1

Manually acquire a clean pair, set `BURST_TARGET_1` to a descriptive field label, and run exactly one bounded five-frame burst. Stop using this field if a beam approaches an ROI edge.

In [ ]:
burst_1 = capture_same_pair_burst(BURST_TARGET_1, 'same_pair_burst_1')


In [ ]:
# MANUAL ACTION: acquire a different clean field, update BURST_TARGET_2, then run once.
burst_2 = capture_same_pair_burst(BURST_TARGET_2, 'same_pair_burst_2')


In [ ]:
# Optional third five-frame burst only while moon background and acquisition remain useful.
# burst_3 = capture_same_pair_burst('field_3', 'same_pair_burst_3')


In [ ]:
# Do not create a fourth burst merely to reach a frame count. Inspect saved frames first.


In [ ]:
# Optional diagnostic only: do not rotate the field rotator for this no-tracking salvage run.


### Optional one-cycle double-ratio attempt

Run only if the same pair survived a five-frame burst with ample edge margin. This is one diagnostic cycle, not a standard-star calibration; do not recenter during it. If any HWP frame loses the pair or clips, label the entire cycle invalid.

In [ ]:
capture_beam_swap_cycle(BURST_TARGET_1, 'optional_same_pair_cycle', 1, None)


In [ ]:
# Do not repeat the HWP cycle unless the first one is demonstrably complete and valid.


In [ ]:
# Remaining time goes to HWP=0° same-pair bursts, not additional unvalidated cycles.


In [ ]:
# No field-rotator repeat tonight: it cannot distinguish sky/instrument polarization without a stable target.


In [ ]:
# Keep the field rotator fixed for the remainder of this salvage sequence.


### Random-field diagnostic capture

Every saved HWP=0° field with a usable pair contributes beam balance, PSF matching, peak-headroom, background, and cadence diagnostics. It does not contribute an empirical repeat S/N unless it belongs to a same-pair burst.

In [ ]:
# Optional: capture_same_pair_burst('field_3', 'same_pair_burst_3')


In [ ]:
# Optional: capture_same_pair_burst('field_4', 'same_pair_burst_4')


In [ ]:
# Stop rather than accumulate unlabeled frames after moon background becomes problematic.


In [ ]:
# End random-field capture before calibration darks.


## 8 · Matching bias and darks

After the stellar blocks, take 20 biases and ten darks at the actual gain-168 science exposure. All must use the active science ROI and opaque Dark slot. The 5/60/150 s ×3 ladder is optional and runs only after these mandatory frames.

In [ ]:
for _ in range(20):
    capture_one(frame_type='BIAS', block='bias20', exposure_s=0.0)

assert SCIENCE_EXP_S is not None and SCIENCE_EXP_S > 0
for _ in range(10):
    capture_one(frame_type='DARK', block='dark_science', exposure_s=SCIENCE_EXP_S)
# Optional only after all mandatory science and calibration frames:
# for exposure_s in (5.0, 60.0, 150.0):
#     for _ in range(3):
#         capture_one(frame_type='DARK', block=f'dark_optional_{exposure_s:g}s', exposure_s=exposure_s)


## 9 · Live checks and handoff

At the telescope, use these only to detect clipping, temperature drift, or missing HWP angles. The final aperture photometry and S/N comparison belong in the dated reduction notebook. There, write one `aperture_measurements.csv` row per FITS frame, beam, and tested aperture radius.

For every saved frame, retain per-beam source counts, sky, PSF width, peak, and detector position. Compute empirical S/N = mean(net source counts) / sample standard deviation only within a labelled same-pair burst; unrelated fields are not repeats. A theoretical electron S/N at gain 168 remains OPEN until its conversion gain and read noise are characterized. Reduce only a complete, same-pair four-angle cycle with `poltools.modulation.double_ratio`; derive q/u uncertainty from complete-cycle scatter. Do not turn diagnostic frames into a science master flat.

In [ ]:
live.session_table(SESSION_DIR)
all_stats = [live.frame_stats(p) for p in sorted(SESSION_DIR.glob('*.fits'))]
live.temperature_trend(all_stats); plt.show()
live.level_trend(all_stats); plt.show()
live.hwp_coverage(SESSION_DIR)


In [ ]:
from obs_utils.night_safety import verify_filter_wheel
verify_filter_wheel(s.imaging)
print('Raw FITS:', SESSION_DIR)
print('Capture ledger:', MANIFEST)
print('Aperture-ledger schema:', APERTURE_LEDGER)
print('Copy the complete session directory to the external drive before leaving.')


## Shutdown

Release only camera, wheel, and HWP. The mount is never touched.

In [ ]:
obs.shutdown()
